# Email Spam Classifier - Model Training and Evaluation

This notebook trains and evaluates multiple ML models for spam classification.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import sys
sys.path.append('..')
from src.feature_engineering import FeatureEngineer
from src.model_training import ModelTrainer

## 1. Load Preprocessed Data

In [ ]:
# Load preprocessed data
df = pd.read_csv('../data/preprocessed_spam.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Feature Extraction using TF-IDF

In [ ]:
# Initialize feature engineer
feature_engineer = FeatureEngineer(method='tfidf', max_features=5000)

# Fit and transform the cleaned messages
X_tfidf = feature_engineer.fit_transform(df['cleaned_message'])
y = df['label']

print(f"TF-IDF Matrix Shape: {X_tfidf.shape}")
print(f"Labels Shape: {y.shape}")

## 3. Split Data

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"\nTraining set distribution:\n{pd.Series(y_train).value_counts()}")
print(f"\nTest set distribution:\n{pd.Series(y_test).value_counts()}")

## 4. Train Multiple Models

In [ ]:
# Initialize trainer
trainer = ModelTrainer()

# Train all models
print("Training models...")
trainer.train_all_models(X_train, y_train)
print("\nAll models trained successfully!")

## 5. Evaluate Models

In [ ]:
# Evaluate all models
results = trainer.evaluate_all_models(X_test, y_test)

# Display results
results_df = trainer.get_results_dataframe()
print("\nModel Results:")
print("=" * 60)
print(results_df)

## 6. Visualize Model Comparison

In [ ]:
# Plot model comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Accuracy comparison
results_df['accuracy'].plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('Model Accuracy Comparison', fontsize=14)
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0.9, 1.0)
axes[0].tick_params(axis='x', rotation=45)

# F1-score comparison
results_df['f1'].plot(kind='bar', ax=axes[1], color='lightgreen', edgecolor='black')
axes[1].set_title('Model F1-Score Comparison', fontsize=14)
axes[1].set_ylabel('F1-Score')
axes[1].set_ylim(0.9, 1.0)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 7. Get Best Model

In [ ]:
# Get best model
best_name, best_model = trainer.get_best_model()
print(f"\nBest Model: {best_name}")
print(f"Accuracy: {results[best_name]['accuracy']:.4f}")
print(f"F1-Score: {results[best_name]['f1']:.4f}")

## 8. Confusion Matrix

In [ ]:
# Confusion matrix for best model
y_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
plt.title(f'Confusion Matrix - {best_name}', fontsize=14)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

## 9. Save Best Model

In [ ]:
# Save the best model
trainer.save_model(best_name, '../models/spam_classifier.pkl')
print(f"\nBest model ({best_name}) saved to '../models/spam_classifier.pkl'")

## 10. Test Predictions

In [ ]:
# Test with sample messages
sample_messages = [
    "Hey, how are you doing today?",
    "CONGRATULATIONS! You've won $1000! Click here to claim NOW!!!",
    "Meeting scheduled for tomorrow at 3pm",
    "FREE FREE FREE! Call now to claim your prize!",
    "Can you pick up groceries on your way home?"
]

# Preprocess and predict
from src.text_preprocessing import TextPreprocessor
preprocessor = TextPreprocessor()

print("Sample Predictions:")
print("=" * 80)
for msg in sample_messages:
    cleaned = preprocessor.preprocess(msg)
    vectorized = feature_engineer.transform([cleaned])
    prediction = best_model.predict(vectorized)[0]
    probability = best_model.predict_proba(vectorized)[0]
    
    print(f"Message: {msg}")
    print(f"Prediction: {'SPAM' if prediction == 1 else 'HAM'}")
    print(f"Confidence: {max(probability)*100:.2f}%")
    print("-" * 80)